# Conversion CZI avec total_bounding_rectangle

Ce notebook reprend la logique du notebook original, mais au lieu de convertir scène par scène avec `scenes_bounding_rectangle`, il utilise `total_bounding_rectangle`.

Donc le résultat correspond à toute la lame CZI dans un seul repère global.

In [ ]:
import os
import numpy as np
from PIL import Image
from pylibCZIrw import czi as pyczi

## 1. Définir les chemins
Modifie `rootdir`, `outdir` et `czifilename`.

In [ ]:
rootdir = "/chemin/vers/le/dossier_czi"
outdir = "/chemin/vers/le/dossier_sortie"
czifilename = "mon_fichier.czi"

os.makedirs(outdir, exist_ok=True)

czifile = os.path.join(rootdir, czifilename)
print(czifile)

## 2. Regarder le bounding box global de la lame

In [ ]:
with pyczi.open_czi(czifile) as czidoc:
    total_bounding_rectangle = czidoc.total_bounding_rectangle

print(total_bounding_rectangle)
print("x =", total_bounding_rectangle.x)
print("y =", total_bounding_rectangle.y)
print("w =", total_bounding_rectangle.w)
print("h =", total_bounding_rectangle.h)

## 3. Fonction de conversion avec total_bounding_rectangle

C'est la même idée que la fonction originale, mais il n'y a plus de boucle sur les scènes.

Avant :
```python
scenes_bounding_rectangle = czidoc.scenes_bounding_rectangle
for i in range(len(scenes_bounding_rectangle)):
    bbox = scenes_bounding_rectangle[i]
```

Maintenant :
```python
bbox = czidoc.total_bounding_rectangle
```

In [ ]:
def convert_czi_total_bbox_to_tiff(pathin, czifilename, pathout,
                                   patch_factor=4,
                                   downsampling_factor=8,
                                   full_patch_w_h=1536,
                                   channels=(0, 1)):
    """
    Convertit toute la lame CZI en utilisant total_bounding_rectangle.

    pathin:
        dossier contenant le CZI

    czifilename:
        nom du fichier CZI

    pathout:
        dossier de sortie

    patch_factor:
        facteur utilisé pour définir la taille du patch lu.
        patch_width = patch_factor * full_patch_w_h

    downsampling_factor:
        1 = pas de downsampling
        8 = garde 1 pixel sur 8

    full_patch_w_h:
        taille de base du patch

    channels:
        canaux à convertir, par exemple (0,) ou (0, 1)
    """

    os.makedirs(pathout, exist_ok=True)

    czifile_scenes = os.path.join(pathin, czifilename)
    base_name = os.path.splitext(os.path.basename(czifilename))[0]

    with pyczi.open_czi(czifile_scenes) as czidoc:

        # ICI : on prend toute la lame, pas les scènes séparées
        bbox = czidoc.total_bounding_rectangle

        print("Total bounding rectangle:")
        print(bbox)
        print("x =", bbox.x, "y =", bbox.y, "w =", bbox.w, "h =", bbox.h)

        patch_width_full = patch_factor * full_patch_w_h
        patch_height_full = patch_factor * full_patch_w_h

        downsampled_patch_w = int(patch_width_full / downsampling_factor)
        downsampled_patch_h = int(patch_height_full / downsampling_factor)

        nb_patch_w = int(bbox.w / patch_width_full)
        nb_patch_h = int(bbox.h / patch_height_full)

        mosaic_image_width = round(float(bbox.w) / downsampling_factor + 0.5)
        mosaic_image_height = round(float(bbox.h) / downsampling_factor + 0.5)

        print("nb_patch_w =", nb_patch_w)
        print("nb_patch_h =", nb_patch_h)
        print("mosaic width =", mosaic_image_width)
        print("mosaic height =", mosaic_image_height)

        for channel in channels:

            print("=" * 60)
            print("Converting channel C" + str(channel))

            mosaic_image = np.zeros(
                (int(mosaic_image_height), int(mosaic_image_width)),
                dtype="uint16"
            )

            for x in range(0, nb_patch_w + 1):
                for y in range(0, nb_patch_h + 1):

                    patch_width = patch_width_full
                    patch_height = patch_height_full

                    if y == nb_patch_h:
                        patch_height = bbox.h - (patch_height_full * y)

                    if x == nb_patch_w:
                        patch_width = bbox.w - (patch_width_full * x)

                    # Si on tombe exactement sur le bord, patch_width ou patch_height peut être 0
                    if patch_width <= 0 or patch_height <= 0:
                        continue

                    my_roi_patched = (
                        bbox.x + patch_width_full * x,
                        bbox.y + patch_height_full * y,
                        patch_width,
                        patch_height,
                    )

                    print("C" + str(channel), "x =", x, "y =", y, "roi =", my_roi_patched)

                    ch = czidoc.read(
                        roi=my_roi_patched,
                        plane={"C": channel}
                    )

                    # czidoc.read retourne souvent (Y, X, 1)
                    ch = np.asarray(ch)
                    if ch.ndim == 3:
                        ch = ch[..., 0]
                    else:
                        ch = np.squeeze(ch)

                    if downsampling_factor == 1:
                        ch_res = ch
                    else:
                        ch_res = ch[::downsampling_factor, ::downsampling_factor]

                    out_y0 = y * downsampled_patch_h
                    out_x0 = x * downsampled_patch_w

                    out_y1 = out_y0 + ch_res.shape[0]
                    out_x1 = out_x0 + ch_res.shape[1]

                    mosaic_image[out_y0:out_y1, out_x0:out_x1] = ch_res

            filename = (
                base_name
                + "_totalbbox_ds"
                + str(downsampling_factor)
                + "_C"
                + str(channel)
                + ".tiff"
            )

            output_path = os.path.join(pathout, filename)

            im = Image.fromarray(mosaic_image.astype(np.uint16))
            im.save(output_path)

            print("Saved:", output_path)

## 4. Lancer la conversion

Pour ne pas downsampler, mets `downsampling_factor=1`.

Attention : si la lame est très grande, `downsampling_factor=1` peut créer un fichier énorme.

In [ ]:
convert_czi_total_bbox_to_tiff(
    pathin=rootdir,
    czifilename=czifilename,
    pathout=outdir,
    patch_factor=4,
    downsampling_factor=8,
    full_patch_w_h=1536,
    channels=(0, 1)
)

## 5. Version sans downsampling

À utiliser seulement si tu as assez de RAM / espace disque.

In [ ]:
# convert_czi_total_bbox_to_tiff(
#     pathin=rootdir,
#     czifilename=czifilename,
#     pathout=outdir,
#     patch_factor=4,
#     downsampling_factor=1,
#     full_patch_w_h=1536,
#     channels=(0,)
# )